<a href="https://colab.research.google.com/github/ozguturgut/RLwOR_forTacticalDM/blob/main/MainTestv5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get install -y glpk-utils coinor-cbc

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  coinor-libcbc3 coinor-libcgl1 coinor-libclp1 coinor-libcoinutils3v5
  coinor-libosi1v5 libamd2 libcolamd2 libglpk40 libsuitesparseconfig5
Suggested packages:
  libiodbc2-dev
The following NEW packages will be installed:
  coinor-cbc coinor-libcbc3 coinor-libcgl1 coinor-libclp1
  coinor-libcoinutils3v5 coinor-libosi1v5 glpk-utils libamd2 libcolamd2
  libglpk40 libsuitesparseconfig5
0 upgraded, 11 newly installed, 0 to remove and 38 not upgraded.
Need to get 3,533 kB of archives.
After this operation, 10.5 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libsuitesparseconfig5 amd64 1:5.10.1+dfsg-4build1 [10.4 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libamd2 amd64 1:5.10.1+dfsg-4build1 [21.6 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 libcolamd2 amd64 1

In [ ]:
!which glpsol
!which cbc

/usr/bin/glpsol
/usr/bin/cbc


In [ ]:
from __future__ import division
from concurrent.futures import ProcessPoolExecutor
from pyomo.environ import *
import csv
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
from itertools import permutations
import pickle
import pdb
import pandas as pd
import pickle
from typing import Any
import QlearnExp_v14_func_SimOpt as Qlearn
import QlearnExp_v14_func_SimOptRedS as QlearnRedS
import Rollout_v4_MC_func_OptRoll_par as RolloutMC   #Rollout_v4_MC_func_OptRoll_par
import kitchenheuristic as Heur
import Analyze as plots
import QlearnExp_v14_func_Greed as Qlearn_base

#Inside
# sim-opt based Q-learn (1)
# rollout parallel Sim  (2)
# heuristic decision  (3)

debug_mode=False
Recipes={}
with open('Recipes.csv', mode='r') as csv_file:
    csv_reader = csv.reader(csv_file)
    line_count = 0
    for row in csv_reader:
        if line_count == 0:
            line_count += 1
            continue
        else:
            Recipes[line_count]=row[1:len(row)]  #int(row[selected_menu][ingredient_no])
            line_count += 1
num_menuoptions=len(Recipes)
Prices={} #for the whole package
with open('Pricesv1.csv', mode='r') as Prices_file:
    Prices_reader = csv.reader(Prices_file)
    line_count = 0
    for row in Prices_reader:
        if line_count == 0:
            line_count += 1
            continue
        else:
            Prices[line_count]=int(row[1][0])
            line_count += 1
num_ingredients=len(Prices)
NutritionalValue={} #for the whole package

with open('Nutrition.csv', mode='r') as Nutrition_file:
    Nutrition_reader = csv.reader(Nutrition_file)
    line_count = 0
    for row in Nutrition_reader:
        if line_count == 0:
            line_count += 1
            continue
        else:
            NutritionalValue[line_count]=int(row[1])
            line_count += 1



#################NEXT ACTION########################################################################################
num_MCepisodes=7 #100 generate num_MCepisodes many random values for price(i.e. stochastic parameter)
lenOfEpisode=15
total_nutrition=0
total_waste=0
total_cost=0
initial_budget=3000
cheapest_menu=8
most_exp_menu=41 #prices are given for pacekage size
max_pack_size=4
usagetime=3
scarcitypricecoeff=3
numberOfIntervals=initial_budget//(cheapest_menu)
menu_with_max_ingredients=13
max_waste=(menu_with_max_ingredients*4)  #in units of quarter package


##########################################################################################################################
last3days=[]
scarce_ingredients=[]
monthly_selection=[]
monthly_selection_details=[]
failed=0
reward_details={}
unused={}
dailywasted=0
monthid=1
xfixes={}

for ingre in range(1,num_ingredients+1):
    for age in range(1, usagetime+1):
        unused[ingre,age]=0 # quantity and age of unused inredient

####################################################################################################################################################################



def save_object(obj: Any, filename: str) -> None:
    """Save Python object to file (binary)."""
    with open(filename, 'wb') as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)


def load_object(filename: str) -> Any:
    """Load Python object from file (binary)."""
    with open(filename, 'rb') as f:
        return pickle.load(f)


################################################################ MAIN  ####################################################################################################
########################################################################################################################################################################################################
if __name__ == "__main__":


    ################################################## test Q-learn Adv #############################################
    trainingDuration=1000
    #exploration_rate=rng.uniform(0, 1.0, (trainingDuration*lenOfEpisode)+100)
    rng2 = np.random.default_rng(seed=399)
    warmstartoff=True
    distribution="uniform"
    numOfEpisodes=45

    '''
    QFactorGrid, QFactorGridPrev, time_to_train=Qlearn.prep_QMatrix(trainingDuration, lenOfEpisode, warmstartoff, distribution, initial_budget)#-----------------------------<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
    save_object(QFactorGrid, "QFactorGrid_adv1500.pkl")

    pdb.set_trace()
    #******************************************
    for i in range(1,4):
        filename="QFactorGrid_adv_"+str(i*500)+".pkl"
        QFactorGrid= load_object(filename)
        summary_Qlearn=Qlearn.simulateQLearn(numOfEpisodes, QFactorGrid, lenOfEpisode, scarcitypricecoeff, scarce_ingredients, distribution, rng2, initial_budget)

        # Save
        filename="summary_Qlearn_adv"+str(i*500)+".pkl"
        save_object(summary_Qlearn, filename)

    df_means=plots.compare_trainDuration()
    QFactorGrid = load_object("QFactorGrid_adv1500.pkl")
    summary_Qlearn=Qlearn.simulateQLearn(numOfEpisodes, QFactorGrid, lenOfEpisode, scarcitypricecoeff, scarce_ingredients, distribution, rng2, initial_budget)
    save_object(summary_Qlearn, "QFactorGrid_adv1500.pkl")

    # Load
    #summary_Qlearn=load_object("summary_Qlearn_adv1500.pkl")
    pdb.set_trace()
    ################################################## test RolloutMC #############################################
    '''
    menus_to_choose=[]
    summary_RollMC, time_to_RollMC=RolloutMC.test_rollout_MC(numOfEpisodes, lenOfEpisode, num_MCepisodes, num_ingredients, usagetime, rng2, distribution,
        num_menuoptions, Recipes, Prices, NutritionalValue, scarce_ingredients, max_pack_size, max_waste, most_exp_menu, initial_budget, menus_to_choose, scarcitypricecoeff)#-----------------------------<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
    # Save
    save_object(summary_RollMC, "summary_RollMC_base.pkl")
    # Load
    #summary_RollMC = load_object("summary_RollMC_base.pkl")
    pdb.set_trace()
    ################################################## test RolloutOpt #############################################
    '''
    summary_Heuristic, time_to_Heuristic=Heur.test_Heuristic(numOfEpisodes, lenOfEpisode, rng2, scarcitypricecoeff, scarce_ingredients, distribution, rng2 , initial_budget)  #-----------------------------<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
    # Save
    save_object(summary_Heuristic, "summary_Heuristic.pkl")
    # Load
    #summary_Heuristic = load_object("summary_Heuristic.pkl")
    #summary_Heuristic=load_object("summary_Heuristic.pkl")
    pdb.set_trace()
    ################################################## test Q-learn base #############################################
    QFactorGrid_base, QFactorGridPrev_base, time_to_train_Qbase=Qlearn_base.prep_QMatrix(trainingDuration, lenOfEpisode, warmstartoff, distribution, initial_budget)#-----------------------------<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
    save_object(QFactorGrid_base, "QFactorGrid_base2500.pkl")
    #pdb.set_trace()
    #QFactorGrid_base=load_object("QFactorGrid_base.pkl")
    #******************************************
    summary_Qlearn_base=Qlearn_base.simulateQLearn(numOfEpisodes, QFactorGrid_base, lenOfEpisode, scarcitypricecoeff, scarce_ingredients, distribution, rng2, initial_budget)
    # Save
    save_object(summary_Qlearn_base, "summary_Qlearn_base.pkl")
    #summary_Qlearn_base=load_object("summary_Qlearn_base2500.pkl")

    ################################################## test Q-learn Reduced States #############################################
    QFactorGrid_reds, QFactorGridPrev_reds, time_to_train_reds=QlearnRedS.prep_QMatrix(trainingDuration, lenOfEpisode, warmstartoff, distribution, initial_budget)#-----------------------------<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
    save_object(QFactorGrid_reds, "QFactorGrid_reds.pkl")
    #pdb.set_trace()
    #QFactorGrid_reds=load_object("QFactorGrid_reds.pkl")
    #******************************************
    summary_Qlearn_reds=QlearnRedS.simulateQLearn(numOfEpisodes, QFactorGrid_reds, lenOfEpisode, scarcitypricecoeff, scarce_ingredients, distribution, rng2, initial_budget)
    # Save
    save_object(summary_Qlearn_reds, "summary_Qlearn_reds1500.pkl")
    #summary_Qlearn_reds=load_object("summary_Qlearn_reds.pkl")
    '''
    ################# RESULTS Baseline ########################################################################################
    #Record the completion time
    pdb.set_trace()
    time_to_train=60*60
    time_to_RollMC=60*60


    plots.plot_bubble(summary_Qlearn, summary_RollMC, summary_Heuristic, summary_Qlearn_base, summary_Qlearn_reds)
    plots.radar_preprint(summary_Qlearn, summary_RollMC, summary_Heuristic, summary_Qlearn_base, summary_Qlearn_reds)
    #plots.plot_lines(summary_Qlearn, summary_RollMC, summary_Heuristic)
    plots.merge_summaries(summary_Qlearn, summary_RollMC, summary_Heuristic, summary_Qlearn_base, summary_Qlearn_reds)
    #print(time_to_train, time_to_RollMC, time_to_RollOpt)
    save_object([time_to_RollMC, time_to_Heuristic], "time_base.pkl")
    plots.plot_box(summary_Qlearn, summary_RollMC, summary_Heuristic, summary_Qlearn_base, summary_Qlearn_reds)

'''
# Convert dict → matrix (keys × iterations)
keys = list(QFactorGridPrev.keys())
max_len = max(len(v) for v in QFactorGridPrev.values())

# Pad with NaN so all lists have equal length
matrix = np.full((len(keys), max_len), np.nan)
for i, k in enumerate(keys):
    vals = QFactorGridPrev[k]
    matrix[i, :len(vals)] = vals

# Heatmap (mask NaN so they don’t show)
vmin = 0    # lower bound of color scale
vmax = 1    # upper bound of color scale
plt.figure(figsize=(12,6))
sns.heatmap(matrix, cmap="viridis", xticklabels=5, yticklabels=False, mask=np.isnan(matrix), vmin=vmin, vmax=vmax)  # force scale
plt.xlabel("Iteration")
plt.ylabel("Keys")
plt.title("Heatmap of QFactorGridPrev values (stabilization check)")
plt.show()
'''






Detected 2 CPU cores
Initial CPU Utilization: 3.0%


In [ ]:
import numpy as np

def generate_beta(alpha, beta, n=1):
    """
    Generate beta distributed random variables using acceptance-rejection method.
    This uses the most common algorithm: Cheng's BC algorithm (1978).
    Works efficiently for all alpha, beta > 0.

    Parameters:
    alpha: first shape parameter (α > 0)
    beta: second shape parameter (β > 0)
    n: number of random variates to generate

    Returns:
    Array of beta distributed random numbers in (0, 1)
    """
    # Setup: Compute constants
    a = alpha + beta

    if min(alpha, beta) <= 1.0:
        # Use simple acceptance-rejection for small parameters
        b = max(1.0/alpha, 1.0/beta)
    else:
        # Cheng's BB algorithm constants
        b = np.sqrt((a - 2.0) / (2.0*alpha*beta - a))

    c = alpha + 1.0/b

    samples = []

    while len(samples) < n:
        # Generate candidates
        U1, U2 = [0.7,0.25]
        # Transform U1
        V = b * np.log(U1 / (1.0 - U1))
        W = alpha * np.exp(V)

        # Compute acceptance threshold
        z = U1**2 * U2
        r = c * V - np.log(4.0)
        s = alpha + r - W

        # Acceptance test (three conditions for efficiency)
        if s + 2.609437912 >= 5.0 * z:  # 2.609... = 1 + log(5)
            # Accept immediately
            X = W / (beta + W)
            samples.append(X)
        elif s >= np.log(z):
            # Second acceptance test
            X = W / (beta + W)
            samples.append(X)
        # Otherwise reject and continue

    return np.array(samples)
# Example usage:
samples = generate_beta(alpha=2.0, beta=5.0, n=1)
print(samples)



[0.40352018]
